In [1]:
# 1. Install dependencies + accelerate
!pip install -q streamlit sentence-transformers transformers tokenizers pandas numpy accelerate
# 2. Install localtunnel
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼
changed 22 packages in 2s
⠼
⠼3 packages are looking for funding
⠼  run `npm fund` for details
⠼

In [2]:
# 1. Clean up any existing directory if running again
!rm -rf economic-news-sentiment

# 2. Clone your clean GitHub repo (replace with your personal GitHub URL)
!git clone https://github.com/Nas-Mohd/economic-news-sentiment.git

# 3. CRITICAL: Move your notebook's working directory into the directory containing app.py
# Use '%cd' (IPython magic) instead of '!cd' (subshell) to keep the working directory active!
%cd economic-news-sentiment/demo

Cloning into 'economic-news-sentiment'...
remote: Enumerating objects: 648, done.
remote: Counting objects: 100% (265/265), done.
remote: Compressing objects: 100% (197/197), done.
remote: Total 648 (delta 152), reused 156 (delta 68), pack-reused 383 (from 1)
Receiving objects: 100% (648/648), 8.81 MiB | 4.03 MiB/s, done.
Resolving deltas: 100% (395/395), done.
Updating files: 100% (75/75), done.
/content/economic-news-sentiment/demo


In [3]:
print("Pre-downloading Hugging Face models to cache (this prevents timeouts)...")
from transformers import pipeline
from sentence_transformers import SentenceTransformer

# 1. Cache Semantic Router
print("\n[1/3] Caching all-MiniLM-L6-v2...")
_ = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# 2. Cache Fine-Tuned DeBERTa (Requires 'accelerate' package)
print("\n[2/3] Caching Fine-Tuned DeBERTa Classifier (takes ~1 minute)...")
_ = pipeline(
    "text-classification",
    model="dummfak/deberta-v3-base-macroeconomic-aspect-classifier",
    device_map="auto"
)

# 3. Cache Fine-Tuned ABSA FinBERT
print("\n[3/3] Caching Fine-Tuned ABSA FinBERT...")
_ = pipeline(
    "text-classification",
    model="dummfak/finbert-macroeconomic-absa"
)

print("\n🎉 All models cached successfully and ready!")

Pre-downloading Hugging Face models to cache (this prevents timeouts)...

[1/3] Caching all-MiniLM-L6-v2...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


[2/3] Caching Fine-Tuned DeBERTa Classifier (takes ~1 minute)...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]


[3/3] Caching Fine-Tuned ABSA FinBERT...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


🎉 All models cached successfully and ready!


In [4]:
# 3. Download the standalone Cloudflare Tunnel client daemon
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared-linux-amd64

In [5]:
# 4. Launch Streamlit in the background, start the Cloudflare Tunnel, and output your direct link
import subprocess
import time
import re

print("🚀 Launching Streamlit in the background...")
# Run Streamlit on port 8501, redirecting output to streamlit.log
with open("streamlit.log", "w") as f_st:
    subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "127.0.0.1"], stdout=f_st, stderr=f_st)

# Wait a brief moment for Streamlit server to spin up
time.sleep(3)

print("🌐 Starting Cloudflare Tunnel...")
# Start free Cloudflare Tunnel proxying port 8501
with open("tunnel.log", "w") as f_tun:
    subprocess.Popen(["./cloudflared-linux-amd64", "tunnel", "--url", "http://127.0.0.1:8501"], stdout=f_tun, stderr=f_tun)

# Allow the tunnel to perform handshake and receive the URL
time.sleep(30)

# Extract the trycloudflare URL automatically from your tunnel.log file
tunnel_url = None
with open("tunnel.log", "r") as f_log:
    log_data = f_log.read()
    matches = re.findall(r'https?://[a-zA-Z0-9-]+\.trycloudflare\.com', log_data)
    if matches:
        tunnel_url = matches[0]

if tunnel_url:
    print("\n" + "="*65)
    print("✨ SUCCESS: Your Economic News Sentiment ABSA App is operational!")
    print(f"🔗 Click this official link: {tunnel_url}")
    print("="*65)
else:
    print("\n⚠️ Tunnel URL extraction timed out.")
    print("Run the command below to locate your generated link:")
    !cat tunnel.log

🚀 Launching Streamlit in the background...
🌐 Starting Cloudflare Tunnel...

✨ SUCCESS: Your Economic News Sentiment ABSA App is operational!
🔗 Click this official link: https://suddenly-words-rental-chubby.trycloudflare.com


In [11]:
import urllib

# Print the bypass IP address
try:
    ip_addr = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
    print("👉 Your Tunnel Password / IP is:", ip_addr)
    print("Copy this IP into the 'Tunnel Password' field when the localtunnel page opens.\n")
except Exception as e:
    print("Could not retrieve IP address:", e)

# Run localtunnel
!npx localtunnel --port 8501

👉 Your Tunnel Password / IP is: 35.198.221.193
Copy this IP into the 'Tunnel Password' field when the localtunnel page opens.

⠙⠹⠸⠼your url is: https://true-points-flash.loca.lt
^C


In [5]:
!pip install gradio transformers torch pandas numpy beautifulsoup4 -q
# 3. Secure drive connection, open port, and boot UI
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
# =====================================================================
# 📦 STEP 1: INITIALIZE DEPENDENCIES & EXPERT CALIBRATED THRESHOLDS
# =====================================================================
import os
import re
import numpy as np
import torch
import torch.nn.functional as F
import pandas as pd
import gradio as gr
from bs4 import BeautifulSoup
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Core Configuration Profiles
ASPECTS = ["Monetary Financial", "Inflation Prices", "Real Economic Activity", "Labor Consumption", "Fiscal Government", "External Sector"]
LABEL_MAPPING = {0: "🟢 Positive", 1: "🔴 Negative", 2: "🟡 Neutral"}

# YOUR EXACT CALIBRATED DEBERTA THRESHOLDS MATRIX
DEBERTA_THRESHOLDS = {
    "Monetary Financial": 0.35,
    "Inflation Prices": 0.50,
    "Real Economic Activity": 0.20,
    "Labor Consumption": 0.40,
    "Fiscal Government": 0.60,
    "External Sector": 0.35
}

# Local Drive Path Registers
DEBERTA_PATH = "/content/drive/MyDrive/economic_news_project/final_deberta_domain_classifier"
FINBERT_FINETUNED_PATH = "/content/drive/MyDrive/economic_news_project/final1_finbert_aspect_sentiment"
BASE_MODEL_NAME = "ProsusAI/finbert"

print("📥 Initializing deep learning architectures...")

# Determine execution device dynamically (GPU if available, else CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️ Routing computing matrix to: {device.upper()}")

def load_classification_assets(path_or_name):
    try:
        tok = AutoTokenizer.from_pretrained(path_or_name)
        mod = AutoModelForSequenceClassification.from_pretrained(path_or_name)
        mod = mod.to(device)
        mod.eval()
        return tok, mod
    except Exception as e:
        print(f"⚠️ Path asset unavailable ({path_or_name}). Falling back to baseline architecture. Error: {e}")
        tok = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
        mod = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL_NAME)
        mod = mod.to(device)
        mod.eval()
        return tok, mod

# Initialize Model Instances
base_tok, base_mod = load_classification_assets(BASE_MODEL_NAME)
deb_tok, deb_mod = load_classification_assets(DEBERTA_PATH)
abs_tok, abs_mod = load_classification_assets(FINBERT_FINETUNED_PATH)

print("🚀 Core neural engines loaded successfully!")

# =====================================================================
# 🧠 STEP 2: ENGINE COMPUTATION LOGIC SHARDS
# =====================================================================

def run_inference(tokenizer, model, text, text_pair=None):
    """Passes strings cleanly and returns pure raw unscaled NumPy logits"""
    with torch.no_grad():
        inputs = tokenizer(
            text=str(text).strip(),
            text_pair=text_pair,
            max_length=128,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        ).to(device)

        outputs = model(**inputs)
        # Pull raw, unaltered network outputs
        raw_logits = outputs.logits.squeeze(0).cpu().numpy()

    return raw_logits

def process_routing(sentence):
    """Page 2: Replicates your training notebook's Sigmoid conversion perfectly"""
    if not sentence.strip(): return "Please enter a valid text slice."

    # 1. Baseline Route: String Match Tracking
    baseline_scores = {}
    for aspect in ASPECTS:
        keywords = set(re.sub(r'[^a-zA-Z ]', '', aspect.lower()).split())
        match_count = sum(1 for word in keywords if word in sentence.lower())
        baseline_scores[aspect] = f"Matches: {match_count}"

    # 2. Fine-Tuned Route: Pull raw logits and run identical manual Sigmoid scaling
    raw_logits = run_inference(deb_tok, deb_mod, sentence)

    # This matches your notebook line: probabilities = 1 / (1 + np.exp(-logits))
    deb_probs = 1 / (1 + np.exp(-raw_logits))

    output_df = []
    for i, aspect in enumerate(ASPECTS):
        prob_val = float(deb_probs[i]) if i < len(deb_probs) else 0.0

        # Pull your exact custom calibrated boundary line
        threshold = DEBERTA_THRESHOLDS[aspect]
        status = "🟢 [ACTIVE]" if prob_val >= threshold else "⚪ [inactive]"

        output_df.append({
            "Macro Aspect Group": aspect,
            "Baseline Keyword Signal": baseline_scores[aspect],
            "DeBERTa Sigmoid Score": f"{prob_val * 100:.1f}%",
            "Assigned Optimal Threshold": f"{threshold * 100:.1f}%",
            "Pipeline Classification Status": status
        })

    return pd.DataFrame(output_df)

def process_sentiment(sentence, aspect):
    """Page 3: Evaluates Context Isolation using side-by-side validation"""
    if not sentence.strip(): return {}, {}

    # Left Column: Standard Base FinBERT
    base_probs = run_inference(base_tok, base_mod, sentence, is_multilabel=False)
    base_res = {LABEL_MAPPING[idx]: float(base_probs[idx]) for idx in range(min(3, len(base_probs)))}

    # Right Column: Your Custom Fine-Tuned ABSA Model (with Text-Pair Aspect Attachment)
    tuned_probs = run_inference(abs_tok, abs_mod, sentence, text_pair=aspect, is_multilabel=False)
    tuned_res = {LABEL_MAPPING[idx]: float(tuned_probs[idx]) for idx in range(min(3, len(tuned_probs)))}

    return base_res, tuned_res

def process_document(raw_doc):
    """Page 4: The Clean Multi-Stage Document Compilation Matrix"""
    if not raw_doc.strip(): return "No text provided."

    # Strip HTML formatting cleanly if present
    if "<html" in raw_doc.lower() or "<div>" in raw_doc.lower() or "<p>" in raw_doc.lower():
        soup = BeautifulSoup(raw_doc, "html.parser")
        clean_text = soup.get_text(separator=" ")
    else:
        clean_text = raw_doc

    # Split text blocks into independent clean evaluation sentences
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', clean_text) if len(s.strip()) > 10]

    records = []
    for sent in sentences:
        # Run sentence through multi-label DeBERTa array
        deb_probs = run_inference(deb_tok, deb_mod, sent, is_multilabel=True)

        detected_aspects = []
        for idx, aspect in enumerate(ASPECTS):
            prob = float(deb_probs[idx]) if idx < len(deb_probs) else 0.0
            # Route checks through your customized optimization boundaries
            if prob >= DEBERTA_THRESHOLDS[aspect]:
                detected_aspects.append(aspect)

        # If aspects breach their specialized limits, run the sentiment scorer on them
        if detected_aspects:
            for active_aspect in detected_aspects:
                sent_probs = run_inference(abs_tok, abs_mod, sent, text_pair=active_aspect, is_multilabel=False)
                winning_idx = np.argmax(sent_probs)

                records.append({
                    "Parsed Sentence Context": sent,
                    "Inferred Macro Aspect": active_aspect,
                    "Directional Sentiment": LABEL_MAPPING[winning_idx],
                    "Confidence Tracking": f"{sent_probs[winning_idx]*100:.1f}%"
                })
        else:
            records.append({
                "Parsed Sentence Context": sent,
                "Inferred Macro Aspect": "None (Abstain)",
                "Directional Sentiment": "░ N/A",
                "Confidence Tracking": "0.0%"
            })

    return pd.DataFrame(records)

# =====================================================================
# 🎨 STEP 3: THE MINIMALIST MONOCHROME INTERFACE LAYOUT
# =====================================================================

with gr.Blocks(theme=gr.themes.Monochrome(font=[gr.themes.GoogleFont("Inter"), "sans-serif"])) as demo:
    gr.Markdown("# 🩺 MULTI-FILE ABSA ENGINE INTERACTIVE SANDBOX")
    gr.Markdown("---")

    with gr.Tabs():
        with gr.TabItem("🎯 Aspect Classification"):
            gr.Markdown("### Evaluates whether sentences breach custom optimized DeBERTa thresholds.")
            input_text_p1 = gr.Textbox(label="Economic Input Headline / Clause", placeholder="Type a sentence here...")
            route_btn = gr.Button("Analyze System Routing Channels", variant="primary")
            output_df_p1 = gr.DataFrame(label="Calibrated Threshold Routing Status Matrix")

            route_btn.click(fn=process_routing, inputs=input_text_p1, outputs=output_df_p1)

        with gr.TabItem("⚖️ Aspect Sentiment Scoring"):
            gr.Markdown("### Compares Native Global FinBERT against your Fine-Tuned text-pair ABSA weights.")
            with gr.Row():
                input_text_p2 = gr.Textbox(label="Target Sentence Context", placeholder="Type targeted phrase here...")
                input_aspect_p2 = gr.Dropdown(choices=ASPECTS, label="Isolate Specific Evaluation Target Aspect", value=ASPECTS[1])

            score_btn = gr.Button("Execute Dual Inferences", variant="primary")

            with gr.Row():
                output_label_base = gr.Label(label="Baseline FinBERT (Net Global Sentence View)")
                output_label_tuned = gr.Label(label="Fine-Tuned ABSA Model (Targeted Context Pair View)")

            score_btn.click(fn=process_sentiment, inputs=[input_text_p2, input_aspect_p2], outputs=[output_label_base, output_label_tuned])

        with gr.TabItem("🚀 Production Document Parsing Engine"):
            gr.Markdown("### Paste a comprehensive macroeconomic article block or raw HTML source code below.")
            input_doc = gr.Textbox(label="Raw Market Article / Page Source Stream", lines=10, placeholder="Paste data here...")
            parse_btn = gr.Button("Parse Document Structure", variant="primary")
            output_df_p3 = gr.DataFrame(label="Decoded Multi-Label Executive Log Table", wrap=True)

            parse_btn.click(fn=process_document, inputs=input_doc, outputs=output_df_p3)

# =====================================================================
# 🏁 STEP 4: FLIP INLINE OFF & SHIFT TO A PUBLIC SECURE TUNNEL LINK
# =====================================================================
# inline=False ensures we completely eliminate the Google Colab white frame glitch.
# share=True creates a beautiful external runtime URL valid for 72 hours.
demo.launch(inline=False, share=True, debug=True)

📥 Initializing deep learning architectures...
🖥️ Routing computing matrix to: CUDA


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

🚀 Core neural engines loaded successfully!


/tmp/ipykernel_5166/3545688591.py:184: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Monochrome(font=[gr.themes.GoogleFont("Inter"), "sans-serif"])) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://bbb529b693e494beda.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7862 <> https://bbb529b693e494beda.gradio.live


In [8]:
# =====================================================================
# 🔍 DEBERTA INTERACTIVE DISCREPANCY DEBUGGING ENGINE (FIXED)
# =====================================================================
import torch
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

# Input Variables Matching Your Notebook Variables Exactly
TEST_SENTENCE = "Consumer price index indicators surged past expectations, signaling persistent structural inflation."
DEBERTA_PATH = "/content/drive/MyDrive/economic_news_project/final_deberta_domain_classifier"
ONLINE_MODEL_NAME = "Microsoft/deberta-v3-base"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"⚙️ Running inspection suite on framework device: {device.upper()}")
print("-" * 80)

# ---------------------------------------------------------------------
# 🧪 INSPECTION 1: TOKENIZER INITIALIZATION VARIANCE
# ---------------------------------------------------------------------
print("🧪 [TEST 1/4] Tokenizer Serialization Validation...")
tok_online = AutoTokenizer.from_pretrained(ONLINE_MODEL_NAME)
tok_drive = AutoTokenizer.from_pretrained(DEBERTA_PATH)

ids_online = tok_online(TEST_SENTENCE, truncation=True, padding="max_length", max_length=128)["input_ids"]
ids_drive = tok_drive(TEST_SENTENCE, truncation=True, padding="max_length", max_length=128)["input_ids"]

tokenizers_match = ids_online == ids_drive
print(f"  👉 Online Tokenizer Vocabulary Match Status: {'✅ IDENTICAL' if tokenizers_match else '❌ MISMATCH'}")
if not tokenizers_match:
    print(f"     ⚠️ WARNING: Your drive tokenizer maps text differently than the online backbone!")

# ---------------------------------------------------------------------
# 🧪 INSPECTION 2: RAW MANUAL MODEL FORWARD INFERENCE (DIRECT PYTORCH)
# ---------------------------------------------------------------------
print("\n🧪 [TEST 2/4] Direct PyTorch Model Matrix Forward Evaluation...")
model = AutoModelForSequenceClassification.from_pretrained(DEBERTA_PATH).to(device)
model.eval()

with torch.no_grad():
    # Evaluate with standard 128 max padding array blocks
    inputs = tok_drive(TEST_SENTENCE, truncation=True, padding="max_length", max_length=128, return_tensors="pt").to(device)
    outputs = model(**inputs)
    direct_logits = outputs.logits.squeeze(0).cpu().numpy()
    direct_probs = 1 / (1 + np.exp(-direct_logits))

print("  👉 Direct PyTorch Sigmoid Probabilities calculated:")
for i, val in enumerate(direct_probs):
    print(f"     Index {i}: {val*100:.2f}%")

# ---------------------------------------------------------------------
# 🧪 INSPECTION 3: HUGGINGFACE TRAINER WRAPPER HARNESS EXTRACTION
# ---------------------------------------------------------------------
print("\n🧪 [TEST 3/4] Hugging Face Dataset + Trainer Loop Emulation...")

# Replicate your exact notebook tokenize_and_format logic structure
eval_df = pd.DataFrame({'text': [TEST_SENTENCE]})
eval_dataset = Dataset.from_pandas(eval_df)
# REMOVED verbose=False to fix the TypeError
tokenized_eval = eval_dataset.map(lambda x: tok_drive(x["text"], truncation=True, padding="max_length", max_length=128), batched=True)

trainer = Trainer(
    model=model,
    args=TrainingArguments(output_dir="./temp_debug", per_device_eval_batch_size=1, fp16=torch.cuda.is_available(), dataloader_pin_memory=False)
)

trainer_predictions = trainer.predict(tokenized_eval)
trainer_logits = trainer_predictions.predictions[0]
trainer_probs = 1 / (1 + np.exp(-trainer_logits))

print("  👉 Trainer Framework Wrapper Probabilities calculated:")
for i, val in enumerate(trainer_probs):
    print(f"     Index {i}: {val*100:.2f}%")

# ---------------------------------------------------------------------
# 🧪 INSPECTION 4: CORE CROSS-COMPARISON SANITY LOG
# =====================================================================
print("\n" + "="*80)
print("📊 FINAL AUDIT LOG SHEET VERDICT")
print("="*80)

variance = np.abs(direct_probs - trainer_probs)
max_variance = np.max(variance)

print(f"🔹 Absolute Maximum Engine Structural Discrepancy: {max_variance:.6f}")

if max_variance < 1e-4:
    print("\n✅ VERDICT: PyTorch Direct Inference and Trainer loops are mathematically identical.")
    print("   The variance you are seeing is 100% caused by a mismatch in your DOMAIN_LABELS index list order!")
    print("   Look closely at the print percentages above. Which index printed EXACTLY 84.4%?")
else:
    print("\n❌ VERDICT: Engine Pipeline Variance Detected!")
    print("   The framework layers are parsing inputs inconsistently between direct calls and dataset blocks.")

⚙️ Running inspection suite on framework device: CUDA
--------------------------------------------------------------------------------
🧪 [TEST 1/4] Tokenizer Serialization Validation...
  👉 Online Tokenizer Vocabulary Match Status: ✅ IDENTICAL

🧪 [TEST 2/4] Direct PyTorch Model Matrix Forward Evaluation...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

  👉 Direct PyTorch Sigmoid Probabilities calculated:
     Index 0: 54.36%
     Index 1: 50.65%
     Index 2: 42.00%
     Index 3: 41.46%
     Index 4: 50.16%
     Index 5: 62.16%

🧪 [TEST 3/4] Hugging Face Dataset + Trainer Loop Emulation...


Map:   0%|          | 0/1 [00:00<?, ? examples/s]

  👉 Trainer Framework Wrapper Probabilities calculated:
     Index 0: 54.36%
     Index 1: 50.65%
     Index 2: 41.99%
     Index 3: 41.46%
     Index 4: 50.16%
     Index 5: 62.17%

📊 FINAL AUDIT LOG SHEET VERDICT
🔹 Absolute Maximum Engine Structural Discrepancy: 0.000075

✅ VERDICT: PyTorch Direct Inference and Trainer loops are mathematically identical.
   The variance you are seeing is 100% caused by a mismatch in your DOMAIN_LABELS index list order!
   Look closely at the print percentages above. Which index printed EXACTLY 84.4%?


In [11]:
import os
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import classification_report, f1_score

# 1. Paths configuration
FINAL_MODEL_PATH = "/content/drive/MyDrive/economic_news_project/final_deberta_domain_classifier"
SPLIT_DIR = "/content/drive/MyDrive/economic_news_project/processed_splits/"
TEST_FILE_PATH = os.path.join(SPLIT_DIR, "pipeline_test.csv")

DOMAIN_LABELS = [
    "Monetary_Financial",
    "Inflation_Prices",
    "Real_Economic_Activity",
    "Labor_Consumption",
    "Fiscal_Government",
    "External_Sector"
]

TUNED_THRESHOLDS = {
    "Monetary_Financial": 0.35,
    "Inflation_Prices": 0.50,
    "Real_Economic_Activity": 0.20,
    "Labor_Consumption": 0.30,
    "Fiscal_Government": 0.60,
    "External_Sector": 0.35
}

print("🔄 Loading pipeline test split and fine-tuned model...")
test_df = pd.read_csv(TEST_FILE_PATH)
tokenizer = AutoTokenizer.from_pretrained(FINAL_MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(FINAL_MODEL_PATH)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()
print(f"✅ Infrastructure locked. Model loaded on {device}. Evaluating {len(test_df)} out-of-sample sentences.\n")

# =====================================================================
# 3. PATCHED GROUND TRUTH EXTRACTION (Handles missing list commas safely)
# =====================================================================
import ast

def parse_space_separated_list(label_string):
    if not isinstance(label_string, str):
        return label_string
    # Strip brackets, split by whatever whitespace exists, and map to integers
    cleaned = label_string.replace("[", "").replace("]", "").strip()
    return [int(x) for x in cleaned.split()]

if 'labels' in test_df.columns:
    print("Parsing labels formatting...")
    y_true = np.array([parse_space_separated_list(x) for x in test_df['labels']])
else:
    y_true = test_df[DOMAIN_LABELS].values.astype(int)

# =====================================================================

# 4. Batch Processing Inference Loop
y_pred_probs = []

print("🏃 Running test inferences...")
with torch.no_grad():
  # Run this right before your "for text in tqdm(...)" block in BOTH places:
    print(f"📊 FIRST ROW SENTENCE SANITY CHECK:")
    print(f"   Text:  {test_df['text'].iloc[0][:60]}...")
    print(f"   Truth: {y_true[0]}")
    for text in tqdm(test_df['text'].tolist(), desc="Processing sentences"):
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=128)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        outputs = model(**inputs)
        probs = torch.sigmoid(outputs.logits).cpu().numpy()[0]
        y_pred_probs.append(probs)

y_pred_probs = np.array(y_pred_probs)

# 5. Apply Customized Threshold Gates
y_pred_binary = np.zeros_like(y_pred_probs)
for idx, domain in enumerate(DOMAIN_LABELS):
    thresh = TUNED_THRESHOLDS[domain]
    y_pred_binary[:, idx] = (y_pred_probs[:, idx] >= thresh).astype(int)

# 6. Generate Metrics Report
print("\n📋 ==================== FINAL OUT-OF-SAMPLE TEST PERFORMANCE ====================")
report = classification_report(
    y_true,
    y_pred_binary,
    target_names=DOMAIN_LABELS,
    digits=4,
    zero_division=0
)
print(report)

final_macro = f1_score(y_true, y_pred_binary, average="macro")
final_micro = f1_score(y_true, y_pred_binary, average="micro")
print("==================================================================================")
print(f"🚀 FINAL SYSTEM OPERATIONAL MACRO F1: {final_macro:.4f}")
print(f"🎯 FINAL SYSTEM OPERATIONAL MICRO F1: {final_micro:.4f}")
print("==================================================================================\n")

🔄 Loading pipeline test split and fine-tuned model...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

✅ Infrastructure locked. Model loaded on cuda. Evaluating 324 out-of-sample sentences.

Parsing labels formatting...
🏃 Running test inferences...
📊 FIRST ROW SENTENCE SANITY CHECK:
   Text:  PETALING JAYA: Dialog Group Bhd is staying focused in the pu...
   Truth: [1 0 1 0 0 1]


Processing sentences: 100%|██████████| 324/324 [00:09<00:00, 33.35it/s]



📋 ==================== FINAL OUT-OF-SAMPLE TEST PERFORMANCE ====================
                        precision    recall  f1-score   support

    Monetary_Financial     0.2778    1.0000    0.4348        90
      Inflation_Prices     0.7500    0.3000    0.4286        40
Real_Economic_Activity     0.5123    1.0000    0.6776       166
     Labor_Consumption     0.2160    1.0000    0.3553        70
     Fiscal_Government     0.0000    0.0000    0.0000        70
       External_Sector     0.3611    1.0000    0.5306       117

             micro avg     0.3468    0.8228    0.4879       553
             macro avg     0.3529    0.7167    0.4045       553
          weighted avg     0.3570    0.8228    0.4624       553
           samples avg     0.3466    0.7692    0.4599       553

🚀 FINAL SYSTEM OPERATIONAL MACRO F1: 0.4045
🎯 FINAL SYSTEM OPERATIONAL MICRO F1: 0.4879



In [10]:
print("🔄 Loading pipeline test split safely...")
# Load the raw file
raw_test_df = pd.read_csv(TEST_FILE_PATH)

# CRITICAL STEP: Explicitly drop any corrupted index row fragments and force-realign the array tracking structures
test_df = raw_test_df.dropna(subset=['text']).reset_index(drop=True)

# Re-extract target columns directly out of this aligned slice
if 'labels' in test_df.columns:
    print("Parsing labels formatting natively...")
    y_true = np.array([parse_space_separated_list(x) for x in test_df['labels']])
else:
    # Explicit column alignment check
    y_true = test_df[DOMAIN_LABELS].values.astype(int)

🔄 Loading pipeline test split safely...
Parsing labels formatting natively...
